# Random Projections for Linear Programming — Toy Demo

This notebook is a compact numerical demonstration of the workflow used in the project:

**Original LP → Random Projection → Projected LP → Solution Retrieval → APM**

We consider the standard-form LP

$$\min_x c^T x \quad \text{s.t.}\quad Ax=b,\;0\le x\le 5.$$

The random projection reduces the number of equality constraints from $m$ to $k<m$. The projected solution is generally not feasible for the original constraints, so we compare a single equality projection with Alternating Projection Method (APM) retrieval. The experiment is illustrative rather than a reproduction of the paper's full benchmark suite.

Revision 2 checks both bounds during retrieval and reporting, uses correctly scaled three-point projections, and includes projection/model construction in wall times. Outputs go to `results/toy_example_v2`. This remains a toy study with a fresh instance per replication, not a QR/Diet table reproduction.


## Setup and overview


In [ ]:
"""
Numerical example for the paper:
'Random Projections for Linear Programming: An Improved Retrieval Phase'

What this script does
---------------------
1. Generate synthetic feasible LP instances in standard form:
       min c^T x
       s.t. A x = b
            0 <= x <= 5
2. Solve the original LP.
3. Build a random projection matrix T and solve the projected LP:
       min c^T x
       s.t. T A x = T b
            0 <= x <= 5
4. Apply two retrieval methods:
   (a) Single projection (Eq. (5) in the paper)
   (b) APM = alternating projection method
5. Compare:
   - runtime
   - equality feasibility ||A x - b||
   - lower and upper bound violations
   - objective gap
6. Plot four figures and save CSV files.

Packages used
-------------
numpy
    Basic array / matrix computation.

pandas
    Save and aggregate experiment results in tables.

matplotlib
    Plot the figures for the presentation.

scipy.optimize.linprog
    Solves LPs numerically. This is our LP solver.

scipy.sparse.linalg.lsmr
    Iterative least-squares / linear-system solver.
    This is included because Section 3.5 of the paper discusses
    approximate pseudoinverse / retrieval using iterative methods.
    In this script you can switch between:
      - exact solve with numpy.linalg.solve
      - approximate solve with scipy.sparse.linalg.lsmr

Important note
--------------
- This is a presentation-friendly synthetic experiment, not a full
  reproduction of all paper benchmarks.
- The script is designed to be readable and easy to modify.
"""



## 1. Import required tools


In [ ]:
# =========================
# 1. Import required tools
# =========================

import time  # Used to measure runtime of each stage.
from pathlib import Path  # Used to create output folders and file paths.

import matplotlib.pyplot as plt  # Used to create plots.
import numpy as np  # Used for arrays, linear algebra, and random generation.
import pandas as pd  # Used to store results in DataFrames / CSV files.
from scipy.optimize import linprog  # Used to solve LPs.
from scipy.sparse.linalg import lsmr  # Used for approximate retrieval updates.




## 2. Global configuration for the whole script


In [ ]:
from pathlib import Path
import sys
GLOBAL_SEED = 42
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / 'src/result_table_generator.py').exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
OUTPUT_DIR = PROJECT_ROOT / 'results' / 'toy_example_v2'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 3. Helper function: random projection T


In [ ]:
# ==========================================
# 3. Helper function: random projection T
# ==========================================


def achlioptas_projection_matrix(k, m, rng):
    """Shared corrected implementation; see src.toy_retrieval."""
    from src.toy_retrieval import achlioptas_projection_matrix as implementation
    return implementation(k, m, rng)





## 4. Helper function: build a feasible LP in standard form


In [ ]:
# =======================================================
# 4. Helper function: build a feasible LP in standard form
# =======================================================


def generate_feasible_lp(m: int, n: int, seed: int):
    """
    Generate a synthetic feasible LP:
        min c^T x
        s.t. A x = b
             0 <= x <= 5

    Construction idea
    -----------------
    1. Randomly generate a nonnegative vector x_true.
    2. Randomly generate a matrix A.
    3. Define b = A x_true.

    Then x_true is automatically feasible.

    Parameters
    ----------
    m : int
        Number of equality constraints.
    n : int
        Number of decision variables.
    seed : int
        Seed for the local RNG.

    Returns
    -------
    A : np.ndarray, shape (m, n)
        Constraint matrix.
    b : np.ndarray, shape (m,)
        Right-hand side vector.
    c : np.ndarray, shape (n,)
        Objective vector.
    x_true : np.ndarray, shape (n,)
        A known feasible point used to build b.
    upper_bound : float
        Common upper bound for all variables in the LP solver.
    """
    # Create a local RNG so each problem instance is reproducible.
    rng = np.random.default_rng(seed)

    # Randomly generate a dense matrix A.
    A = rng.standard_normal((m, n))

    # Create a sparse-ish nonnegative feasible vector x_true.
    x_true = np.zeros(n)

    # Select exactly m active coordinates to make x_true easier to interpret.
    active_idx = rng.choice(n, size=m, replace=False)

    # Assign positive values on the chosen active coordinates.
    x_true[active_idx] = rng.uniform(0.2, 2.0, size=m)

    # Define b = A x_true so that x_true is feasible by construction.
    b = A @ x_true

    # Create a nonnegative objective vector c.
    c = rng.uniform(0.0, 1.0, size=n)

    # Set a simple common upper bound for the LP variables.
    upper_bound = 5.0

    # Return everything needed for the LP.
    return A, b, c, x_true, upper_bound




## 5. Helper function: compute performance metrics


In [ ]:
# ============================================
# 5. Helper function: compute performance metrics
# ============================================


def compute_metrics(A, b, c, x, upper_bound=5.0, delta=1e-06):
    """Shared corrected implementation; see src.toy_retrieval."""
    from src.toy_retrieval import compute_metrics as implementation
    return implementation(A, b, c, x, upper_bound, delta)





## 6. Solve the original LP (baseline method)


In [ ]:
def solve_original_lp(A, b, c, upper_bound):
    """Wall time includes solver input preparation and optimization."""
    start_time = time.perf_counter()
    result = linprog(c, A_eq=A, b_eq=b, bounds=[(0., upper_bound)] * A.shape[1], method='highs')
    if not result.success:
        raise RuntimeError(f'Original LP failed: {result.message}')
    return result.x.copy(), time.perf_counter() - start_time, result


## 7. Solve the projected LP after random projection


In [ ]:
def solve_projected_lp(A, b, c, k, upper_bound, seed):
    """Wall time includes T, TA/Tb, model construction, solve and cleanup."""
    start_time = time.perf_counter()
    T = achlioptas_projection_matrix(k, A.shape[0], np.random.default_rng(seed))
    x, _, result = solve_original_lp(T @ A, T @ b, c, upper_bound)
    return x, T, time.perf_counter() - start_time, result


## 8. Retrieval helper: exact equality projection update


In [ ]:
# =====================================================
# 8. Retrieval helper: exact equality projection update
# =====================================================


def exact_equality_projection_update(A, b, x):
    """Shared corrected implementation; see src.toy_retrieval."""
    from src.toy_retrieval import exact_equality_projection_update as implementation
    return implementation(A, b, x)





## 9. Retrieval helper: approximate equality projection update


In [ ]:
# ==========================================================
# 9. Retrieval helper: approximate equality projection update
# ==========================================================


def approximate_equality_projection_update(A, b, x):
    """Shared corrected implementation; see src.toy_retrieval."""
    from src.toy_retrieval import approximate_equality_projection_update as implementation
    return implementation(A, b, x)





## 10. Single projection retrieval (one-shot)


In [ ]:
# =============================================
# 10. Single projection retrieval (one-shot)
# =============================================


def single_projection_retrieval(
    A: np.ndarray,
    b: np.ndarray,
    x_proj: np.ndarray,
    mode: str = "exact",
):
    """
    Apply a single equality projection step to the projected LP solution.

    This corresponds to a one-shot retrieval:
        projected solution -> Eq. (5) once

    Parameters
    ----------
    A, b : np.ndarray
        Original LP data.
    x_proj : np.ndarray
        Solution of the projected LP.
    mode : str
        "exact"      -> use numpy.linalg.solve
        "approx_lsmr"-> use scipy.sparse.linalg.lsmr

    Returns
    -------
    x_single : np.ndarray
        Retrieved solution after one equality projection.
    runtime : float
        Runtime of this one-shot retrieval step.
    """
    # Record start time.
    start_time = time.perf_counter()

    # Choose the update method.
    if mode == "exact":
        x_single = exact_equality_projection_update(A, b, x_proj)
    elif mode == "approx_lsmr":
        x_single = approximate_equality_projection_update(A, b, x_proj)
    else:
        raise ValueError("mode must be 'exact' or 'approx_lsmr'")

    # Measure runtime.
    runtime = time.perf_counter() - start_time

    # Return the retrieved point and runtime.
    return x_single, runtime




## 11. Alternating Projection Method (APM) retrieval


In [ ]:
# ======================================================
# 11. Alternating Projection Method (APM) retrieval
# ======================================================


def apm_retrieval(A, b, x_start, mode='exact', max_iter=30, delta=1e-06, upper_bound=5.0):
    """Shared corrected implementation; see src.toy_retrieval."""
    from src.toy_retrieval import apm_retrieval as implementation
    return implementation(A, b, x_start, mode, max_iter, delta, upper_bound)





## 12. Run the whole experiment over multiple sizes


In [ ]:
# ===================================================
# 12. Run the whole experiment over multiple sizes
# ===================================================


def run_experiment(
    sizes,
    n_factor: int = 2,
    projection_ratio: float = 0.4,
    replications: int = 5,
    retrieval_mode: str = "exact",
):
    """
    Run the full numerical experiment for multiple LP sizes.

    Parameters
    ----------
    sizes : list[int]
        Different values of m (number of equality constraints).
    n_factor : int
        Number of variables is n = n_factor * m.
    projection_ratio : float
        Projected size is k = projection_ratio * m.
    replications : int
        Number of random instances per size.
    retrieval_mode : str
        "exact" or "approx_lsmr"

    Returns
    -------
    runs_df : pd.DataFrame
        Raw results for all runs.
    apm_trace_df : pd.DataFrame
        Per-iteration APM traces across all runs.
    summary_df : pd.DataFrame
        Average results aggregated by problem size.
    """
    # Create lists to accumulate run-level and iteration-level records.
    run_rows = []
    trace_rows = []

    # Loop over different original constraint sizes m.
    for m in sizes:
        # Set the number of variables n relative to m.
        n = n_factor * m

        # Set projected dimension k.
        k = max(4, int(round(projection_ratio * m)))

        # Repeat the experiment several times to average out randomness.
        for rep in range(replications):
            # Create a deterministic seed for this instance.
            seed = GLOBAL_SEED + 100 * m + rep

            # Generate a feasible LP instance.
            A, b, c, x_true, upper_bound = generate_feasible_lp(m=m, n=n, seed=seed)

            # Solve the original LP.
            x_original, time_original, _ = solve_original_lp(A, b, c, upper_bound)

            # Solve the projected LP.
            x_projected, T, time_projected, _ = solve_projected_lp(
                A, b, c, k, upper_bound, seed=seed + 999
            )

            # Apply one-shot single retrieval.
            x_single, time_single = single_projection_retrieval(
                A=A,
                b=b,
                x_proj=x_projected,
                mode=retrieval_mode,
            )

            # Apply APM retrieval.
            x_apm, time_apm, history_df = apm_retrieval(
                A=A,
                b=b,
                x_start=x_projected,
                mode=retrieval_mode,
                max_iter=30,
                delta=1e-6,
                upper_bound=upper_bound,
            )

            # Compute all metrics.
            metrics_original = compute_metrics(A, b, c, x_original, upper_bound=upper_bound)
            metrics_projected = compute_metrics(A, b, c, x_projected, upper_bound=upper_bound)
            metrics_single = compute_metrics(A, b, c, x_single, upper_bound=upper_bound)
            metrics_apm = compute_metrics(A, b, c, x_apm, upper_bound=upper_bound)

            # Compute relative objective gaps versus the original LP optimum.
            obj_base = max(1e-8, abs(metrics_original["objective"]))

            projected_rel_gap = abs(metrics_projected["objective"] - metrics_original["objective"]) / obj_base
            single_rel_gap = abs(metrics_single["objective"] - metrics_original["objective"]) / obj_base
            apm_rel_gap = abs(metrics_apm["objective"] - metrics_original["objective"]) / obj_base

            # Store the run-level results.
            run_rows.append(
                {
                    "m": m,
                    "n": n,
                    "k": k,
                    "replication": rep,
                    "instance_seed": seed,
                    "projection_seed": seed + 999,
                    "upper_bound": upper_bound,
                    "original_status": "OPTIMAL",
                    "projected_status": "OPTIMAL",
                    "retrieval_mode": retrieval_mode,
                    "time_original": time_original,
                    "time_projected": time_projected,
                    "time_single": time_single,
                    "time_apm": time_apm,
                    "time_projected_plus_apm": time_projected + time_apm,
                    "ratio_projected_vs_original": time_projected / time_original,
                    "ratio_projected_plus_apm_vs_original": (time_projected + time_apm) / time_original,
                    "original_objective": metrics_original["objective"],
                    "projected_objective": metrics_projected["objective"],
                    "single_objective": metrics_single["objective"],
                    "apm_objective": metrics_apm["objective"],
                    "projected_eq_residual": metrics_projected["eq_residual"],
                    "single_eq_residual": metrics_single["eq_residual"],
                    "apm_eq_residual": metrics_apm["eq_residual"],
                    "projected_negativity": metrics_projected["negativity_sum"],
                    "single_negativity": metrics_single["negativity_sum"],
                    "apm_negativity": metrics_apm["negativity_sum"],
                    "projected_rel_obj_gap": projected_rel_gap,
                    "single_rel_obj_gap": single_rel_gap,
                    "apm_rel_obj_gap": apm_rel_gap,
                    "projected_upper_violation_sum": metrics_projected["upper_violation_sum"],
                    "projected_bound_violation_sum": metrics_projected["bound_violation_sum"],
                    "projected_max_bound_violation": metrics_projected["max_bound_violation"],
                    "projected_feasible": metrics_projected["feasible"],
                    "single_upper_violation_sum": metrics_single["upper_violation_sum"],
                    "single_bound_violation_sum": metrics_single["bound_violation_sum"],
                    "single_max_bound_violation": metrics_single["max_bound_violation"],
                    "single_feasible": metrics_single["feasible"],
                    "apm_upper_violation_sum": metrics_apm["upper_violation_sum"],
                    "apm_bound_violation_sum": metrics_apm["bound_violation_sum"],
                    "apm_max_bound_violation": metrics_apm["max_bound_violation"],
                    "apm_feasible": metrics_apm["feasible"],
                    "apm_iterations_recorded": len(history_df),
                }
            )

            # Store the APM iteration trace, if any iterations were recorded.
            if len(history_df) > 0:
                history_copy = history_df.copy()
                history_copy["m"] = m
                history_copy["n"] = n
                history_copy["k"] = k
                history_copy["replication"] = rep
                history_copy["retrieval_mode"] = retrieval_mode
                trace_rows.append(history_copy)

    # Combine run-level rows into one DataFrame.
    runs_df = pd.DataFrame(run_rows)

    # Combine trace rows into one DataFrame.
    if len(trace_rows) > 0:
        apm_trace_df = pd.concat(trace_rows, ignore_index=True)
    else:
        apm_trace_df = pd.DataFrame()

    # Aggregate averages by m for presentation-quality summary tables.
    summary_df = (
        runs_df.groupby(["m", "retrieval_mode"], as_index=False)
        .agg(
            time_original=("time_original", "mean"),
            time_projected=("time_projected", "mean"),
            time_single=("time_single", "mean"),
            time_apm=("time_apm", "mean"),
            time_projected_plus_apm=("time_projected_plus_apm", "mean"),
            ratio_projected_vs_original=("ratio_projected_vs_original", "mean"),
            ratio_projected_plus_apm_vs_original=("ratio_projected_plus_apm_vs_original", "mean"),
            projected_eq_residual=("projected_eq_residual", "mean"),
            single_eq_residual=("single_eq_residual", "mean"),
            apm_eq_residual=("apm_eq_residual", "mean"),
            projected_negativity=("projected_negativity", "mean"),
            single_negativity=("single_negativity", "mean"),
            apm_negativity=("apm_negativity", "mean"),
            projected_rel_obj_gap=("projected_rel_obj_gap", "mean"),
            single_rel_obj_gap=("single_rel_obj_gap", "mean"),
            apm_rel_obj_gap=("apm_rel_obj_gap", "mean"),
            projected_upper_violation_sum=("projected_upper_violation_sum", "mean"),
            projected_bound_violation_sum=("projected_bound_violation_sum", "mean"),
            projected_feasible_runs=("projected_feasible", "sum"),
            single_upper_violation_sum=("single_upper_violation_sum", "mean"),
            single_bound_violation_sum=("single_bound_violation_sum", "mean"),
            single_feasible_runs=("single_feasible", "sum"),
            apm_upper_violation_sum=("apm_upper_violation_sum", "mean"),
            apm_bound_violation_sum=("apm_bound_violation_sum", "mean"),
            apm_feasible_runs=("apm_feasible", "sum"),
            apm_iterations_recorded=("apm_iterations_recorded", "mean"),
        )
    )

    # Return all result tables.
    return runs_df, apm_trace_df, summary_df




## 13. Plotting helper for presentation figures


In [ ]:
# ===========================================
# 13. Plotting helper for presentation figures
# ===========================================


def make_plots(summary_df: pd.DataFrame, retrieval_mode: str, output_dir: Path):
    """
    Create and save four presentation-friendly figures.

    Parameters
    ----------
    summary_df : pd.DataFrame
        Aggregated summary results.
    retrieval_mode : str
        Either "exact" or "approx_lsmr", used in filenames.
    output_dir : Path
        Folder where the figures will be saved.

    Returns
    -------
    plot_paths : dict
        Dictionary mapping plot names to their saved file paths.
    """
    # Filter summary rows for the chosen retrieval mode.
    data = summary_df[summary_df["retrieval_mode"] == retrieval_mode].copy()

    # Sort by m so the x-axis is ordered nicely.
    data = data.sort_values("m")

    # This dictionary will store the final file paths.
    plot_paths = {}

    # ------------------------------------------
    # Plot 1: Runtime comparison
    # ------------------------------------------
    plt.figure(figsize=(8, 5))
    plt.plot(data["m"], data["time_original"], marker="o", label="Original LP")
    plt.plot(data["m"], data["time_projected"], marker="o", label="Projection + projected LP")
    plt.plot(data["m"], data["time_projected_plus_apm"], marker="o", label="Projected LP + APM")
    plt.xlabel("Number of original constraints m")
    plt.ylabel("Average wall time (seconds)")
    plt.title(f"Runtime comparison ({retrieval_mode})")
    plt.legend()
    plt.tight_layout()

    runtime_path = output_dir / f"runtime_comparison_{retrieval_mode}.png"
    plt.savefig(runtime_path, dpi=200)
    plt.close()

    plot_paths["runtime"] = runtime_path

    # ------------------------------------------
    # Plot 2: Equality feasibility
    # ------------------------------------------
    plt.figure(figsize=(8, 5))
    plt.plot(data["m"], data["projected_eq_residual"], marker="o", label="Projected solution")
    plt.plot(data["m"], data["single_eq_residual"], marker="o", label="Single projection")
    plt.plot(data["m"], data["apm_eq_residual"], marker="o", label="APM")
    plt.yscale("symlog", linthresh=1e-12)
    plt.xlabel("Number of original constraints m")
    plt.ylabel("Average ||A x - b||")
    plt.title(f"Equality-feasibility improvement ({retrieval_mode})")
    plt.legend()
    plt.tight_layout()

    feasibility_path = output_dir / f"feasibility_improvement_{retrieval_mode}.png"
    plt.savefig(feasibility_path, dpi=200)
    plt.close()

    plot_paths["feasibility"] = feasibility_path

    # ------------------------------------------
    # Plot 3: Nonnegativity violation
    # ------------------------------------------
    plt.figure(figsize=(8, 5))
    plt.plot(data["m"], data["projected_bound_violation_sum"], marker="o", label="Projected solution")
    plt.plot(data["m"], data["single_bound_violation_sum"], marker="o", label="Single projection")
    plt.plot(data["m"], data["apm_bound_violation_sum"], marker="o", label="APM")
    plt.yscale("symlog", linthresh=1e-12)
    plt.xlabel("Number of original constraints m")
    plt.ylabel("Average total lower + upper bound violation")
    plt.title(f"Full-bound feasibility ({retrieval_mode})")
    plt.legend()
    plt.tight_layout()

    negativity_path = output_dir / f"bound_feasibility_{retrieval_mode}.png"
    plt.savefig(negativity_path, dpi=200)
    plt.close()

    plot_paths["nonnegativity"] = negativity_path

    # ------------------------------------------
    # Plot 4: Relative objective gap
    # ------------------------------------------
    plt.figure(figsize=(8, 5))
    plt.plot(data["m"], data["projected_rel_obj_gap"], marker="o", label="Projected solution")
    plt.plot(data["m"], data["single_rel_obj_gap"], marker="o", label="Single projection")
    plt.plot(data["m"], data["apm_rel_obj_gap"], marker="o", label="APM")
    plt.xlabel("Number of original constraints m")
    plt.ylabel("Average relative objective gap")
    plt.title(f"Objective gap (see feasibility counts) ({retrieval_mode})")
    plt.legend()
    plt.tight_layout()

    obj_gap_path = output_dir / f"objective_gap_{retrieval_mode}.png"
    plt.savefig(obj_gap_path, dpi=200)
    plt.close()

    plot_paths["objective_gap"] = obj_gap_path

    # Return the saved paths.
    return plot_paths




## 14. Main script entry point


In [ ]:
# ===========================
# 14. Main script entry point
# ===========================


def main():
    """
    Main function:
    1. Set experiment sizes and parameters.
    2. Run experiments for exact retrieval.
    3. Run experiments for approximate LSMR retrieval.
    4. Save CSV files and figures.
    5. Print a compact summary to the terminal.
    """
    # Choose several original constraint sizes m.
    sizes = [20, 30, 40, 50, 60]

    # Set n = n_factor * m.
    n_factor = 2

    # Set projected size k = projection_ratio * m.
    projection_ratio = 0.4

    # Number of random instances per size.
    replications = 5

    # Run the experiment with exact retrieval updates.
    runs_exact, trace_exact, summary_exact = run_experiment(
        sizes=sizes,
        n_factor=n_factor,
        projection_ratio=projection_ratio,
        replications=replications,
        retrieval_mode="exact",
    )

    # Run the experiment with approximate LSMR retrieval updates.
    runs_lsmr, trace_lsmr, summary_lsmr = run_experiment(
        sizes=sizes,
        n_factor=n_factor,
        projection_ratio=projection_ratio,
        replications=replications,
        retrieval_mode="approx_lsmr",
    )

    # Combine exact and approximate summaries into unified tables.
    runs_df = pd.concat([runs_exact, runs_lsmr], ignore_index=True)
    summary_df = pd.concat([summary_exact, summary_lsmr], ignore_index=True)

    # Combine the APM iteration traces.
    if len(trace_exact) > 0 and len(trace_lsmr) > 0:
        apm_trace_df = pd.concat([trace_exact, trace_lsmr], ignore_index=True)
    elif len(trace_exact) > 0:
        apm_trace_df = trace_exact.copy()
    elif len(trace_lsmr) > 0:
        apm_trace_df = trace_lsmr.copy()
    else:
        apm_trace_df = pd.DataFrame()

    # Save CSV files.
    runs_path = OUTPUT_DIR / "all_runs.csv"
    summary_path = OUTPUT_DIR / "summary_by_size.csv"
    trace_path = OUTPUT_DIR / "apm_trace.csv"

    runs_df.to_csv(runs_path, index=False)
    summary_df.to_csv(summary_path, index=False)
    apm_trace_df.to_csv(trace_path, index=False)

    # Record the experimental contract alongside the v2 results.
    import json, platform
    from importlib.metadata import version
    metadata = {
        'revision': 2, 'backend': 'scipy-highs', 'python': platform.python_version(),
        'libraries': {name: version(name) for name in ('numpy', 'scipy', 'pandas')},
        'global_seed': GLOBAL_SEED, 'instance_seed': 'GLOBAL_SEED + 100*m + replication',
        'projection_seed': 'instance_seed + 999', 'bounds': [0, 5],
        'projection': 'three-point, nonzero probability 1/3, scale sqrt(3/k)',
        'retrieval': 'ordinary APM; exact Eq.(5) or LSMR correction on A; full bounds',
        'timing': 'wall: original includes solver setup/solve/cleanup; projected includes T, TA/Tb and solver setup/solve/cleanup; projected_plus_apm adds retrieval; instance generation excluded',
        'claim': 'illustrative random LP experiment; not a structured paper benchmark',
        'sizes': sizes, 'n_factor': n_factor, 'projection_ratio': projection_ratio,
        'replications': replications,
    }
    (OUTPUT_DIR / 'metadata.json').write_text(json.dumps(metadata, indent=2) + '\n', encoding='utf-8')

    # Make figures for exact retrieval.
    exact_plot_paths = make_plots(summary_df, retrieval_mode="exact", output_dir=OUTPUT_DIR)

    # Make figures for approximate LSMR retrieval.
    lsmr_plot_paths = make_plots(summary_df, retrieval_mode="approx_lsmr", output_dir=OUTPUT_DIR)

    # Print a compact summary to the terminal for quick inspection.
    print("\n=== Summary table ===")
    print(summary_df.round(6).to_string(index=False))

    print("\n=== Saved CSV files ===")
    print(runs_path)
    print(summary_path)
    print(trace_path)

    print("\n=== Saved exact-mode plots ===")
    for name, path in exact_plot_paths.items():
        print(f"{name}: {path}")

    print("\n=== Saved approx_lsmr-mode plots ===")
    for name, path in lsmr_plot_paths.items():
        print(f"{name}: {path}")


# This standard Python pattern means:
# if you run this file directly, execute main().
if __name__ == "__main__":
    main()
